# Проект по дисциплине "Теория конечных графов" группа№3

Ладнюк Кира

Гареев Эльдар

Егоров Иван

# 0. Подготовка.

Импортируем необходимые библиотеки

In [514]:
import os
from collections import defaultdict, deque
import random
import numpy as np
from random import randint, random, sample, choice

Напишем функцию для обработки неориентированных графов. Графы будем хранить в виде словаря. При возникновении ошибки будет выходить сообщение об ошибке.

In [515]:
def make_graph(path):
    # Создаем словарь, который по умолчанию будет присваивать ключу пустое множество
    graph = defaultdict(set)
    unique_edges = 0
    all_edges = 0

    try:
        with open(path, 'r') as file:
            for line in file:
                # Убираем лишние символы в начале и конце строки
                line = line.strip()

                # Пропускаем комментарии
                if line.startswith('#') or not line:
                    continue

                u, v = line.split()
                u = int(u)
                v = int(v)

                # Считаем общее количество ребер (включая кратные)
                all_edges += 1

                # Если ребро (u, v) ещё не было добавлено в словарь, добавляем (u, v) и (v, u)
                if v not in graph[u]:

                    graph[u].add(v)
                    graph[v].add(u)

                    # Считаем уникальные ребра
                    unique_edges += 1


    except FileNotFoundError:
        print(f"ошибка: файл {path} не найден.")
        return None

    except Exception as e:
        print(f"произошла ошибка при обработке файла: {e}")
        return None

    return  dict(graph), unique_edges,  all_edges

Напишем функцию для генерации случайных графов. В качестве аргументов будем передавать желаемое количество графов:

In [516]:
def generate_graphs(n=0):
    if not n:
        n = randint(5, 15)
    graphs = []

    for _ in range(n):
        # Выбираем число вершин, равномерно распределенное от 20 до 50
        v = randint(20, 50)
        vertices = [i for i in range(v)]
        # Выбираем количество ребер как максимально возможное, умноженное на коэффициент от 0 до 1
        e = int(((v * (v - 1)) / 2) * random())
        graph = {vertice: set() for vertice in vertices}
        while e:
            u, v = sample(vertices, 2)
            if v not in graph[u]:
                graph[u].add(v)
                graph[v].add(u)
                e -= 1
        graphs.append(graph)
    return graphs

Будем использовать файл WikiVote в качестве примера:

In [517]:
file = "./Wiki-Vote.txt"
graph, edges, _ = make_graph(file)

# Анализ структуры сети

### _Задание 1, часть А_

Для каждой из сетей определить следующие характеристики:
Число вершин, число рёбер, плотность (отношение числа рёбер к максимально возможному числу рёбер), число компонент слабой связности, долю вершин в максимальной по мощности компоненте слабой связности. Для ориентированных графов определить число компонент сильной связности и долю вершин графа в наибольшей компоненте сильной связности компоненте.

Начнем с числа вершин и ребер:
  - кол-во вершин = кол-во ключей в словаре
  - кол-во ребер =  каждое ребро (A, B) хранится дважды, тогда общее количество рёбер можно получить как сумма мощностей множеств, делённая на 2
  - плотность = отношение числа рёбер к максимально возможному числу рёбер

In [518]:
num_vertex = len(graph)
max_num_vertex = num_vertex * (num_vertex - 1) // 2
density = edges / max_num_vertex if max_num_vertex > 0 else 0.0

print(f"Кол-во нод: {num_vertex}, кол-во ребер: {edges}, плотность: {density}")

Кол-во нод: 7115, кол-во ребер: 100762, плотность: 0.003981420144693063


todo: сделать визуализацию


Поиск компонент слабой связности. Для этого:
1. Реализуем DFS с помощью стека
2. Запускаем DFS по всем ключам словаря.
3. Функция вернет списков из списков, где каждый элемент - комппонента слабой связности.

In [519]:
def dfs_stack(graph):

    visited = set()
    components = []

    for u in graph:

        if u not in visited:
            stack = [u]
            component = []

            while stack:
                cur = stack.pop()

                if cur not in visited:
                    visited.add(cur)
                    component.append(cur)

                    for v in graph[cur]:
                        if v not in visited:
                            stack.append(v)

            components.append(component)
    return components


Рассмотрим сколько компонент слабой связности у нас получилось: для этого выведем длину получишегося массива.

In [520]:
components = dfs_stack(graph)
print(len(components))

24


Найдем долю вершин в максимальной по мощности компоненте слабой связности:
1. Найдем компоненту с максимальным числом вершин
2. Поделим мощность компоненты с максимальным числом вершин на общее количество вершин

In [521]:
max_size = max(len(c) for c in components)
fraction = max_size / num_vertex
print(fraction)

0.9931131412508785


### _Задание А часть 2_

Для наибольшей компоненты слабой связности оценить значения диаметра сети, 90 процентиля расстояния (геодезического) между вершинами графа. Оценку провести на основании:

a) Двойного прохода BFS (the double sweep): Для случайно выбранного узла найти
максимально удаленный узел _a_, а затем найти узел _b_, максимально удаленный от _a_. За диаметр принять эксцентриситет вершины _ecc(a) = d(b,a)_

Для решения этого пункта:

1. Найдем наибольшую компоненту слабой связности
2. Используем алгоритм BFS
3. Выберем случайную вершину u, найдем самую удаленную от нее вершину v (первый проход BFS).
4. Найдем самую удаленную от v вершину w (второй проход BFS)

Тогда расстояние d(v, w) будет приближенным диаметром

1. Функция для нахождения самой большой компоненты слабой связности. На выход возвращает подграф в виде словаря.

todo: Эльдар проверь все ли верно

In [522]:
def subgraph(components, graph):
    comp = max(components, key=len)
    nodes = set(comp)
    ans = {}

    # Переносим в ответ данные о тех ребрах, которые соединяют вершины наибольшей компоненты слабой связности
    for node in comp:
        ans[node] = {i for i in graph.get(node, set()) if i in nodes}

    return ans

In [523]:
large_comp = subgraph(components, graph)
print(len(large_comp))

7066


Реализация алгоритма BFS на очереди. Функция возвращает самую дальнюю вершину от выбранной и расстояния до всех вершин компоненты

In [524]:
def bfs_far(graph, u):

    dist = {u: 0}
    queue = deque([u])
    u = u
    max_dist = 0

    while queue:

        cur = queue.popleft()

        for v in graph.get(cur, set()):
            if v not in dist:

                dist[v] = dist[cur] + 1
                queue.append(v)
                if dist[v] > max_dist:
                    u = v
                    max_dist = dist[v]

    return u, dist


3 + 4. Выбираем две случайные вершины и прогоняем два раза bfs на них

In [525]:
def dbl_swp_diam(graph):
    if not graph:
        return 0

    u = choice(list(graph))
    v, _ = bfs_far(graph, u)
    _, ans = bfs_far(graph, v)

    return max(ans.values())

In [526]:
diameter = dbl_swp_diam(large_comp)
diameter

7

Вычисление 90-процентиля расстояний между случайными вершинами:
  1. Выбираем 1000 или 500 (по умолчанию) случайных пар вершин
  2. Для каждой пары делаем BFS, реализованный раннее
  3. Сортируем расстояния и находим 90-процентиль.

In [527]:
def percentile90(graph, n=500):

    nodes = list(graph.keys()) if hasattr(graph, 'keys') else list(graph)

    distances = []

    for _ in range(n):
        u, v = sample(nodes, 2)

        _, d = bfs_far(graph, u)

        if v in d:
            distances.append(d[v])

    return np.percentile(distances, 90) if distances else 0

In [528]:
np.random.seed(26)
def snowball(g, n=500):
    if not g: return {}

    s = sample(list(g), min(3, len(g)))
    v, q = set(s), deque(s)

    while q and len(v) < n:
        c = q.popleft()
        for i in g.get(c, []):
            if i not in v and len(v) < n:
                v.add(i)
                q.append(i)

    sg = {node: set() for node in v}
    for node in v:
        sg[node] = {nb for nb in g.get(node, []) if nb in v}

    return sg

def snowball_1(graph, n=500):
    a = snowball(graph, n)
    b = dbl_swp_diam(a)
    perc = percentile90(a, n=100)
    return b, perc

In [529]:
ans = percentile90(large_comp)
ans

ans

np.float64(4.0)

In [530]:
sn_dim, sn_per = snowball_1(large_comp)
sn_dim,sn_per

(5, np.float64(3.0))

[van]Посчитаем количество треугольников




In [531]:
def triangles(graph):

    triangles = 0

    for u in graph:
        current = graph[u]

        for v in current:

            if v > u:

                for w in current & graph[v]:

                    if w > v:

                        triangles += 1
    return triangles

In [532]:
print(triangles(graph))

608389


[van]degrees

In [533]:
def calculate_node_degrees(graph):

    degrees = [len(u) for u in graph.values()]

    return  min(degrees),  max(degrees), sum(degrees) / len(degrees)

In [534]:
print(calculate_node_degrees(graph))

(1, 1065, 28.32382290934645)


In [535]:
def abba(graph, n):

    nodes = graph.get(n, set())
    kol_vo = len(nodes)
    if kol_vo < 2: return 0.0

    max = kol_vo * (kol_vo - 1) / 2
    unique = 0

    for u in nodes:
        for v in nodes:
            if u > v and v in graph.get(u, set()):
                unique += 1

    return unique / max

def avg_clast(graph):

    ans = 0.0
    count = 0

    for node in graph:
        coeff = abba(graph, node)
        ans += coeff
        count += 1

    return ans / count

In [536]:
print(avg_clast(graph))

0.14089784589308738


In [537]:
def gcc(g):
    t, ct = 0, 0

    for u in g:
        n = g[u]
        k = len(n)

        if k < 2: continue
        t += k * (k - 1) / 2

        for v in n:
            for w in n:

                if v > w and w in g.get(v,set()):
                    ct += 1

    return ct/t if t else 0.0

In [538]:
print(gcc(graph))

0.12547914899233995


In [539]:
def avg_clust(g, comp):
    if not comp: return 0.0

    total = 0.0
    nodes = set(comp)

    for u in comp:
        nb = [v for v in g.get(u, set()) if v in nodes]
        k = len(nb)
        if k < 2: continue

        edges = 0
        for i in range(k):
            for j in range(i+1, k):
                if nb[j] in g.get(nb[i], set()):
                    edges += 1

        total += (2 * edges) / (k * (k - 1))

    return total / len(comp)

In [540]:

def process_directed_graph_file(path):
    graph = defaultdict(set)

    unique_edges = 0
    all_edges = 0

    try:
        with open(path, 'r') as file:
            for line in file:
                line = line.strip()

                if line.startswith('#') or not line:
                    continue

                u, v = line.split()
                u = int(u)
                v = int(v)


                all_edges += 1

                if u not in graph[v]:
                    graph[u].add(v)
                    unique_edges += 1

    except FileNotFoundError:
        print(f"ошибка: файл {path} не найден.")
        return None

    except Exception as e:
        print(f"произошла ошибка при обработке файла: {e}")
        return None

    return dict(graph), unique_edges, all_edges

In [541]:
file = "./Wiki-Vote.txt"
graph, edges, _ = make_graph(file)

In [542]:
print(edges)

100762


In [543]:
def count_scc(graph):
    # Первый проход DFS для определения порядка завершения
    visited = set()
    order = []

    for u in graph:
        if u not in visited:
            stack = [(u, False)]

            while stack:
                cur, processed = stack.pop()
                if processed:
                    order.append(cur)
                    continue
                if cur in visited:
                    continue

                if cur not in visited:
                    visited.add(cur)
                    stack.append((cur, True))
                    for v in graph.get(cur, set()):
                        if v not in visited:
                            stack.append((v, False))

    # Инвертируем граф
    reversed_graph = defaultdict(set)
    for src in graph:
        for dst in graph[src]:
            reversed_graph[dst].add(src)

    # Второй проход DFS в обратном порядке по инвертированному графу
    visited = set()
    components = []

    for u in reversed(order):
        if u not in visited:
            stack = [u]
            visited.add(u)
            component = []

            while stack:
                cur = stack.pop()
                component.append(cur)

                for v in reversed_graph.get(cur, set()):
                    if v not in visited:
                        stack.append(v)
                        visited.add(v)

            components.append(component)
    return components

In [544]:
scc = count_scc(graph)

print(len(scc))

24
